✅ Étape 2 : Extraction automatique des descripteurs temporels (aire segmentée)
🎯 Objectif

Extraire pour chaque image masquée du Groupe 1 les descripteurs statistiques suivants :

    Aire minimale

    Aire maximale

    Moyenne et écart-type des aires

    Pente montante maximale (diastole)

    Pente descendante maximale (systole)

Ces descripteurs permettront d’analyser quantitativement le comportement cardiaque à travers le cycle.

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd

# 📁 Dossier des images segmentées du Groupe 1
segmented_dir = "/home/amenacer/Stage/Data/Segmentation/resultas/tollsome 2D+T/maskedimg/Tollsome_output1"

# 📁 Dossier de sortie pour les features
features_output_dir = "/home/amenacer/Stage/Data/Segmentation/resultas/tollsome 2D+T/features_masked/Tollsome_output1"
os.makedirs(features_output_dir, exist_ok=True)

# 📊 Tableau de résumé global
summary_data = []

# 🔁 Traitement de chaque fichier
for filename in sorted(os.listdir(segmented_dir)):
    if filename.endswith(".nii.gz"):
        file_path = os.path.join(segmented_dir, filename)

        # Charger l'image segmentée
        img = nib.load(file_path)
        data = img.get_fdata()

        # Calcul des aires (pixels > 0)
        aires = np.array([np.sum(data[:, :, t] > 0) for t in range(data.shape[-1])])

        # Calcul des descripteurs
        min_area = np.min(aires)
        max_area = np.max(aires)
        mean_area = np.mean(aires)
        std_area = np.std(aires)
        ascending_slope = np.max(np.diff(aires)) if len(aires) > 1 else 0
        descending_slope = np.min(np.diff(aires)) if len(aires) > 1 else 0

        # Sauvegarde individuelle
        features = np.array([min_area, max_area, mean_area, std_area, ascending_slope, descending_slope])
        feature_path = os.path.join(features_output_dir, filename.replace(".nii.gz", "_features.txt"))
        np.savetxt(feature_path, features,
                   header="min_area max_area mean_area std_area ascending_slope descending_slope",
                   fmt="%.4f")

        print(f"✅ Descripteurs extraits : {filename}")

        # Ajouter au résumé
        summary_data.append({
            "Fichier": filename,
            "Min_Area": min_area,
            "Max_Area": max_area,
            "Mean_Area": mean_area,
            "Std_Area": std_area,
            "Ascending_Slope": ascending_slope,
            "Descending_Slope": descending_slope
        })

# Export du fichier CSV global
summary_df = pd.DataFrame(summary_data)
summary_csv_path = os.path.join(features_output_dir, "features_summary_groupe1.csv")
summary_df.to_csv(summary_csv_path, index=False)
print("\n📊 Fichier résumé sauvegardé :", summary_csv_path)
